# Sample assignment

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/12-assignment/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本節在 CPU 逐節執行並保存輸出；合成資料短訓練、L4 與部署紀錄見網站驗證頁。真實場景長訓練仍待安排。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.3.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 逐節範例預設 CPU；L4 訓練與部署的實測另見網站驗證頁。保留相同版本的 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""Simplified task-aligned top-k assignment, with conflict and empty-GT handling."""
import torch
from torch import nn
from torch.nn import functional as F


def box_iou(boxes_a, boxes_b):
    intersection = (torch.minimum(boxes_a[:, None, 2:], boxes_b[None, :, 2:])
                    - torch.maximum(boxes_a[:, None, :2], boxes_b[None, :, :2])).clamp(min=0).prod(-1)
    area_a = (boxes_a[:, 2:] - boxes_a[:, :2]).prod(-1)
    area_b = (boxes_b[:, 2:] - boxes_b[:, :2]).prod(-1)
    return intersection / (area_a[:, None] + area_b[None] - intersection)


@torch.no_grad()
def assign(points, gt, scores, overlaps, k=2):
    # points [P,2], gt [G,4], scores/overlaps [G,P]; -1 denotes background.
    owner = torch.full((len(points),), -1, dtype=torch.long)
    if len(gt) == 0:
        return owner, torch.zeros_like(overlaps)
    inside = ((points[None] > gt[:, None, :2]) & (points[None] < gt[:, None, 2:])).all(-1)
    metric = scores * overlaps.square()  # teaching alpha=1,beta=2; not original defaults
    selected = torch.zeros_like(inside)
    for g in range(len(gt)):
        eligible = torch.where(inside[g] & (metric[g] > 0))[0]
        n = min(k, len(eligible))
        if n:
            selected[g, eligible[metric[g, eligible].topk(n).indices]] = True
    for p in range(len(points)):
        candidates = torch.where(selected[:, p])[0]
        if len(candidates):
            owner[p] = candidates[overlaps[candidates, p].argmax()]
    return owner, metric * inside


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    points = torch.tensor([[8., 8.], [16., 8.], [24., 8.], [40., 8.]])
    gt = torch.tensor([[0., 0., 20., 16.], [12., 0., 32., 16.]])
    scores = torch.tensor([[.9, .7, .1, .8], [.1, .8, .7, .9]])
    predicted_boxes = torch.tensor([[0., 0., 10., 16.], [4., 0., 24., 16.],
                                    [14., 0., 32., 16.], [40., 0., 50., 10.]])
    overlaps = box_iou(gt, predicted_boxes)  # actual shared boxes, not arbitrary incompatible IoUs
    owner, metric = assign(points, gt, scores, overlaps)
    assert owner.tolist() == [0, 0, 1, -1]
    empty, _ = assign(points, gt[:0], scores[:0], overlaps[:0])
    assert empty.tolist() == [-1] * 4
    logits = nn.Parameter(torch.zeros(4))
    optimizer = torch.optim.SGD([logits], lr=1.)
    target = (owner >= 0).float()  # simplified binary foreground, not TAL's quality targets
    optimizer.zero_grad()
    loss = F.binary_cross_entropy_with_logits(logits, target)
    loss.backward()
    assert torch.allclose(logits.grad, torch.tensor([-.125, -.125, -.125, .125]))
    optimizer.step()
    print('inside-masked score * IoU^2:', metric.round(decimals=3).tolist())
    print('candidate owner (-1 background):', owner.tolist())
    print('foreground target:', target.tolist())
    print('gradient:', logits.grad.tolist())
    print('empty image owner:', empty.tolist())
    moved_boxes = predicted_boxes.clone()
    moved_boxes[1] = gt[1]
    moved_owner, _ = assign(points, gt, scores, box_iou(gt, moved_boxes))
    assert moved_owner.tolist() == [0, 1, 1, -1]
    print('exercise: move predicted box p1 to GT B; owner:', moved_owner.tolist())


if __name__ == '__main__':
    main()


inside-masked score * IoU^2: [[0.22499999403953552, 0.3109999895095825, 0.0, 0.0], [0.0, 0.1469999998807907, 0.5669999718666077, 0.0]]
candidate owner (-1 background): [0, 0, 1, -1]
foreground target: [1.0, 1.0, 1.0, 0.0]
gradient: [-0.125, -0.125, -0.125, 0.125]
empty image owner: [-1, -1, -1, -1]
exercise: move predicted box p1 to GT B; owner: [0, 1, 1, -1]
